In [ ]:
!pip install monai[itk] SimpleITK einops nibabel tqdm

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import random
from pathlib import Path

import torch
import numpy as np
import monai.transforms as mt

from torch.utils.data import DataLoader
from monai.data import Dataset
from monai.networks.nets import SwinUNETR
from monai.losses import DiceLoss
from monai.metrics import DiceMetric
from monai.transforms import AsDiscrete
from monai.inferers import sliding_window_inference
from tqdm import tqdm

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
IMAGES_DIR = Path("/content/drive/MyDrive/panther/ImagesTr")
LABELS_DIR = Path("/content/drive/MyDrive/panther/LabelsTr")
OUTPUT_DIR = Path("/content/drive/MyDrive/SwinUNETR_monai_models_corrected")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(len(list(IMAGES_DIR.glob("*.mha"))))
print(len(list(LABELS_DIR.glob("*.mha"))))

92
92


In [ ]:
def clean(p):
    name = p.name.replace(".nii.gz", "").replace(".mha", "")
    if name.endswith("_0000"):
        name = name[:-5]
    return name

images = sorted(IMAGES_DIR.glob("*.mha"))
labels = sorted(LABELS_DIR.glob("*.mha"))

label_map = {clean(p): p for p in labels}

data = []
for img in images:
    stem = clean(img)
    if stem in label_map:
        data.append({"image": str(img), "label": str(label_map[stem])})

print("Paired:", len(data))

random.seed(42)
random.shuffle(data)

val_size = max(1, int(0.15 * len(data)))
val_data = data[:val_size]
train_data = data[val_size:]

print("Train:", len(train_data))
print("Val:", len(val_data))

Paired: 92
Train: 79
Val: 13


In [ ]:
train_tfms = mt.Compose([
    mt.LoadImaged(keys=["image", "label"], reader="ITKReader"),
    mt.EnsureChannelFirstd(keys=["image", "label"]),
    mt.Spacingd(keys=["image", "label"], pixdim=(1.0, 1.0, 1.0), mode=("bilinear", "nearest")),
    mt.Orientationd(keys=["image", "label"], axcodes="RAS"),
    mt.NormalizeIntensityd(keys=["image"]),

    mt.RandCropByPosNegLabeld(
        keys=["image", "label"],
        label_key="label",
        spatial_size=(96, 96, 96),
        pos=3,
        neg=1,
        num_samples=4,
    ),

    mt.RandFlipd(keys=["image", "label"], prob=0.2, spatial_axis=0),
    mt.RandFlipd(keys=["image", "label"], prob=0.2, spatial_axis=1),
    mt.RandFlipd(keys=["image", "label"], prob=0.2, spatial_axis=2),
    mt.RandRotate90d(keys=["image", "label"], prob=0.2, max_k=3),
    mt.RandScaleIntensityd(keys=["image"], factors=0.1, prob=0.2),
    mt.RandShiftIntensityd(keys=["image"], offsets=0.1, prob=0.2),
])

val_tfms = mt.Compose([
    mt.LoadImaged(keys=["image", "label"], reader="ITKReader"),
    mt.EnsureChannelFirstd(keys=["image", "label"]),
    mt.Spacingd(keys=["image", "label"], pixdim=(1.0, 1.0, 1.0), mode=("bilinear", "nearest")),
    mt.Orientationd(keys=["image", "label"], axcodes="RAS"),
    mt.NormalizeIntensityd(keys=["image"]),
])

monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.


In [ ]:
train_ds = Dataset(train_data, transform=train_tfms)
val_ds = Dataset(val_data, transform=val_tfms)

train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, num_workers=3)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=1)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))

Train batches: 40
Val batches: 13


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

model = SwinUNETR(in_channels=1, out_channels=3, feature_size=48, spatial_dims=3).to(device)

weight = torch.hub.load_state_dict_from_url(
    "https://github.com/Project-MONAI/MONAI-extra-test-data/releases/download/0.8.1/swin_unetr.base_5000ep_f48_lr2e-4_pretrained.pt"
)

if "state_dict" in weight:
    weight = weight["state_dict"]

model_dict = model.state_dict()
pretrained_dict = {}

for k, v in weight.items():
    k_clean = k.replace("module.", "")

    if k_clean.startswith("swinViT."):
        k_clean = k_clean.replace("swinViT.", "swinViT.")

    if k_clean in model_dict and model_dict[k_clean].shape == v.shape:
        pretrained_dict[k_clean] = v

print("Loaded pretrained tensors:", len(pretrained_dict), "/", len(model_dict))

model_dict.update(pretrained_dict)
model.load_state_dict(model_dict)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=5e-5,
    weight_decay=1e-5,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=200,
    eta_min=1e-6,
)
loss_fn = DiceLoss(
    to_onehot_y=True,
    softmax=True,
    weight=torch.tensor([0.1, 0.6, 0.3]).to(device),
)
dice_metric = DiceMetric(include_background=False, reduction="mean_batch")
post_pred = AsDiscrete(argmax=True, to_onehot=3)
post_label = AsDiscrete(to_onehot=3)

Device: cuda
Downloading: "https://github.com/Project-MONAI/MONAI-extra-test-data/releases/download/0.8.1/swin_unetr.base_5000ep_f48_lr2e-4_pretrained.pt" to /root/.cache/torch/hub/checkpoints/swin_unetr.base_5000ep_f48_lr2e-4_pretrained.pt


100%|██████████| 244M/244M [00:11<00:00, 23.2MB/s]

Loaded pretrained tensors: 157 / 159


In [ ]:
def unpack_patches(batch):
    if isinstance(batch, list):
        images = torch.cat([b["image"] for b in batch], dim=0)
        labels = torch.cat([b["label"] for b in batch], dim=0)
    else:
        images = batch["image"]
        labels = batch["label"]
    return images.to(device), labels.to(device)

In [ ]:
def validate_full_volume():
    model.eval()
    dice_metric.reset()

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Full-volume validation"):
            image = batch["image"].to(device)
            label = batch["label"].to(device)

            logits = sliding_window_inference(
                image,
                roi_size=(96, 96, 96),
                sw_batch_size=1,
                predictor=model,
                overlap=0.5,
                mode="gaussian",
            )

            pred = post_pred(logits[0])
            lab = post_label(label[0])

            dice_metric(
                y_pred=pred.unsqueeze(0),
                y=lab.unsqueeze(0),
            )

    dice_per_class = dice_metric.aggregate()
    dice_metric.reset()

    dice_tumor = dice_per_class[0].item()

    dice_pancreas = dice_per_class[1].item()

    return dice_tumor, dice_pancreas

In [ ]:
def train(epochs):
    best_tumor_dice = -1
    best_path = OUTPUT_DIR / "SwinUNETR_scratch_fullval_best.pth"

    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0
        steps = 0

        for batch in tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}"):
            image, label = unpack_patches(batch)

            optimizer.zero_grad()
            output = model(image)
            loss = loss_fn(output, label)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            steps += 1

        scheduler.step()

        avg_loss = epoch_loss / max(1, steps)

        dice_tumor, dice_pancreas = validate_full_volume()

        print(
            f"Epoch {epoch}/{epochs} — "
            f"Loss: {avg_loss:.4f} — "
            f"Full Dice tumor: {dice_tumor:.4f} — "
            f"Full Dice pancreas: {dice_pancreas:.4f}"
        )

        if dice_tumor > best_tumor_dice:
            best_tumor_dice = dice_tumor
            torch.save(model.state_dict(), best_path)
            print(f"Saved best model: {best_path} tumor Dice={best_tumor_dice:.4f}")

    print("Best full-volume tumor Dice:", best_tumor_dice)

In [ ]:
train(epochs=200)

Full-volume validation:   0%|          | 0/13 [00:00<?, ?it/s]Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
Full-volume validation: 100%|██████████| 13/13 [01:18<00:00,  6.06s/it]


Epoch 1/200 — Loss: 0.2998 — Full Dice tumor: 0.0057 — Full Dice pancreas: 0.0128
Saved best model: /content/drive/MyDrive/SwinUNETR_monai_models_corrected/SwinUNETR_scratch_fullval_best.pth tumor Dice=0.0057


Full-volume validation: 100%|██████████| 13/13 [01:18<00:00,  6.06s/it]


Epoch 2/200 — Loss: 0.2866 — Full Dice tumor: 0.0083 — Full Dice pancreas: 0.0159
Saved best model: /content/drive/MyDrive/SwinUNETR_monai_models_corrected/SwinUNETR_scratch_fullval_best.pth tumor Dice=0.0083


Full-volume validation: 100%|██████████| 13/13 [01:18<00:00,  6.06s/it]


...

Epoch 198/200 — Loss: 0.1402 — Full Dice tumor: 0.2993 — Full Dice pancreas: 0.3920


Full-volume validation: 100%|██████████| 13/13 [01:18<00:00,  6.06s/it]


Epoch 199/200 — Loss: 0.1434 — Full Dice tumor: 0.2986 — Full Dice pancreas: 0.4005


Full-volume validation: 100%|██████████| 13/13 [01:18<00:00,  6.06s/it]

Epoch 200/200 — Loss: 0.1493 — Full Dice tumor: 0.3002 — Full Dice pancreas: 0.3955
Best full-volume tumor Dice: 0.3119204342365265
